In [ ]:
import pandas as pd
import numpy as np
import requests

def fetch_lbins(item_ids):
    def get_price(item):
        data = requests.get(
            f"https://sky.coflnet.com/api/auctions/tag/{item}/active/bin"
        ).json()
        return data[0]["startingBid"] if data else 0

    return {item: get_price(item) for item in item_ids}

bazaar = {
    k: {
        "Sell": v["quick_status"]["sellPrice"],
        "Buy": v["quick_status"]["buyPrice"]
    }
    for k, v in requests.get("https://api.hypixel.net/v2/skyblock/bazaar").json()['products'].items()
}

def apply_price_overrides(df,price_map,weight_col="Weight",price_col="Price"):
    df[price_col] *= 1.35 #Coin bonus from snowman mask and gold gift talisman.

    mask = df["Item"].isin(price_map)
    df.loc[mask, price_col] = df.loc[mask, "Item"].map(price_map)

    weights = df[weight_col]
    factor = weights / weights.sum()

    df[price_col] *= factor

#This function assumes you're buy-ordering the gifts and selling them for current LBIN
def expected_profit(color, filename, price_map):
    df = pd.read_csv(filename)
    apply_price_overrides(df, price_map)
    profit = 2 * df["Price"].sum() - bazaar[f"{color.upper()}_GIFT"]["Sell"]
    return profit

lbin_items = {
    'SNOW_SUIT_HELMET',
    'SNOW_SUIT_CHESTPLATE',
    'SNOW_SUIT_LEGGINGS',
    'SNOW_SUIT_BOOTS',
    'GIFT_THE_FISH',
    'GOLD_GIFT',
    'NEW_BOTTLE_OF_JYRRE',
    'PET_SNOWMAN',
    'CRYOPOWDER_SHARD',
    'WINTER_ISLAND'
}
lbin_prices = fetch_lbins(lbin_items)

snow_minion_cost = bazaar['SNOW_BLOCK']['Sell'] * 992 + bazaar['ENCHANTED_SNOW_BLOCK']['Sell'] * 248
lbin_prices["Snow Minion"] = 250000 - snow_minion_cost

files = {
    "white": "white.csv",
    "green": "green.csv",
    "red": "red.csv"
}

for color, file in files.items():
    profit = expected_profit(color, file, lbin_prices)
    print(f"Expected Profit For {color.capitalize()}: {profit:,.0f}")

Expected Profit For White: 3,119
Expected Profit For Green: 10,505
Expected Profit For Red: 24,870
